# Figure 17 — you have a 96-well plate, not one flask

Taking the top q points of the acquisition function gives q nearly identical experiments. Thompson sampling gives q diverse ones for free, because each draw has its own maximum.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

In [ ]:

xs = np.linspace(0, 10, 700)
OX = np.array([1.15, 2.90, 4.30, 6.10])
OY = land.f1d(OX)
g = gpmod.GP(gpmod.matern52, ls=0.85, sf=1.0, sn=0.03).fit(OX[:, None], OY)
mu, sd = g.predict(xs[:, None])
a = gpmod.ei(mu, sd, OY.max())
Q = 6

# naive: the q highest values of the acquisition function
naive = xs[np.argsort(a)[-Q:]]

# Thompson: q posterior draws, each maximised
draws = g.sample(xs[:, None], n=Q, seed=11)
thom = xs[draws.argmax(axis=1)]

fig, axes = plt.subplots(2, 1, figsize=(5.3, 4.5), sharex=True,
                         gridspec_kw=dict(hspace=0.18))

ax = axes[0]
ax.plot(xs, a, color=style.RED, lw=1.7)
ax.fill_between(xs, 0, a, color=style.RED, alpha=0.12, lw=0)
for x in naive:
    ax.axvline(x, color=style.RED, lw=1.0, alpha=0.85)
ax.set_ylabel("EI")
ax.set_title(f"naive: the top {Q} points of EI — spread {np.ptp(naive):.2f} in x",
             loc="left", fontsize=10.5, color=style.RED)
ax.text(0.985, 0.52, "six wells answering\nthe same question",
        transform=ax.transAxes, ha="right", fontsize=9.6, color=style.RED, bbox=dict(fc="white", alpha=0.85, ec="none", pad=2.0))

ax = axes[1]
for d, x in zip(draws, thom):
    ax.plot(xs, d, color=style.BLUE, lw=0.9, alpha=0.55)
    ax.axvline(x, color=style.BLUE, lw=1.0, alpha=0.85)
    ax.plot([x], [d.max()], "v", ms=7, color=style.BLUE)
ax.plot(OX, OY, "o", ms=6, color=style.INK, mec="white", mew=1.0, zorder=6)
ax.set_ylabel("posterior draws")
ax.set_xlabel("reaction parameter  x")
ax.set_title(f"Thompson sampling: {Q} draws, {Q} maxima — spread "
             f"{np.ptp(thom):.2f} in x", loc="left", fontsize=10.5,
             color=style.BLUE)
print("naive spread  :", round(float(np.ptp(naive)), 3))
print("Thompson spread:", round(float(np.ptp(thom)), 3))
style.save(fig, "fig_17_batch_selection", OUT)